# Stage 00 — Environment and data preflight

This notebook is the mandatory entry point for the Ecuador 2022 workflow. It checks that the approved `pypsa-earth-ec` environment is active, resolves the complete external-artifact dependency set, verifies cached files, downloads missing published files, and validates each file before it becomes authoritative.

The stage performs no scientific transformation and no network optimization.

## Inputs and outputs

| Type | Artifact | Role |
|---|---|---|
| Configuration | `config/artifacts.yaml` | Versioned URIs, destinations, sizes, hashes, stage dependencies, and structural checks |
| External input | `base.nc` | Upstream PyPSA-Earth Ecuador grid |
| External input | `solar.nc` | Hourly 2013 solar profiles |
| External input | `onwind.nc` | Hourly 2013 onshore-wind profiles |
| Output | console report | One status per manifest artifact; no generated scientific dataset |

The files are published under CC BY 4.0 in Zenodo record [`10.5281/zenodo.23392049`](https://doi.org/10.5281/zenodo.23392049). The full ERA5 cutout and locally generated cleaned/assembled networks are intentionally excluded.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src').is_dir():
    raise RuntimeError('Could not locate the repository root containing src/.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.artifacts import format_preflight_report, run_preflight
from src.environment import check_environment, format_environment_report

environment_report = check_environment()
print(format_environment_report(environment_report))
if not environment_report.ok:
    raise RuntimeError('Environment preflight failed. Activate pypsa-earth-ec and rerun.')

artifact_report = run_preflight(stage='00_preflight', root=project_root)
print(format_preflight_report(artifact_report))
artifact_report.raise_for_errors()

## Interpretation

A passing report means that every currently published external input needed by the staged workflow exists at its authoritative repository-relative destination and matches its pinned size, SHA-256, and NetCDF structure. A failure stops the workflow before scientific processing.

Run the command-line equivalent with:

```powershell
python -m src.preflight --stage 00_preflight
```

Use `--offline` to validate the cache without network access. To execute this notebook from a shell without installing a persistent Jupyter kernelspec, use `python -m src.notebook_runner notebooks/00_preflight_and_data_inventory.ipynb`. The runner launches the notebook with the current approved interpreter. Later notebooks call the same preflight logic with their own stage identifier.

## Next stage

After this stage passes, continue to `01_prepare_2022_demand.ipynb` when that Phase 3 notebook is available. Passing preflight confirms data integrity and availability; it does not validate the 2022 model or its carrier composition.